# GECKO v2 (Rust) + C `gecko_fast` Whole-Genome Benchmark — Human (`GRCh38.p14`) vs. Chimpanzee (`Clint_PTRv2`)

Builds the unified Rust **`gecko`** binary (`rust-rewrite` branch: AVX2/SSE2 vectorized 2-bit/bp `PackedSeq` + zero-copy 2-bit GPU + zero-spill slab streaming + 4 MB fast I/O) **and** the C **`gecko_fast`** binary (`gpu-backends` branch) on Google Colab, then benchmarks all **24 homologous Human vs. Chimpanzee chromosome pairs** (`chr1`–`chr22`, `chrX`, `chrY`, pairing Human `chr2` against Chimpanzee `chr2A + chr2B`):

- **`cpu` (Rust)**: 2-bit/bp `PackedSeq` + single canonical Reference-Y Hash-CSR/Direct-CSR + 64-wide software prefetch + 2-tier Fibonacci diagonal cache + AVX2/BMI2 SIMD extension + zero-spill Reference-Slab Streaming
- **`c_fast` (C)**: multi-threaded C `gecko_fast` (`-O3 -march=native`) Hash-CSR + AVX2 CPU baseline
- **`wgpu` (Rust WGPU)**: Cross-vendor zero-copy 2-bit Vulkan/Metal/DX12 compute shader backend (where adapter binding limits permit)
- **`cuda` (Rust + CUDA/CUB)**: Native NVIDIA CUDA + CUB GPU backend with zero-copy 2-bit `PackedSeq`, word-level 2-bit revcomp, 1-bit `--freq-limit` dropped-position bitmap, adaptive sub-tiling, and 32-bp warp-broadcast extension

**Before running:** Select `Runtime -> Change runtime type -> T4 GPU` (or `L4` / `A100`).

In [ ]:
# 1. Verify GPU, CUDA/Vulkan environment, and Rust toolchain
!nvidia-smi || true
!nvcc --version || true
!which vulkaninfo && vulkaninfo --summary | head -n 30 || true
!which cargo || (curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y)
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version && nproc && free -h

In [ ]:
# 2. Clone `rust-rewrite` (Rust gecko) + `gpu-backends` (C gecko_fast) and build both
import os, subprocess

REPO_URL = 'https://github.com/otorreno/gecko.git'
BRANCH = 'rust-rewrite'

!rm -rf /content/gecko /content/gecko_c
!git clone --depth 1 -b {BRANCH} {REPO_URL} /content/gecko
%cd /content/gecko

# Include sm_75 (T4), sm_80 (A100), and sm_89 (L4) so all Colab GPUs run natively
env = os.environ.copy()
has_nvcc = subprocess.run(['which', 'nvcc'], capture_output=True).returncode == 0
if has_nvcc:
    env['GECKO_CUDA_ARCH'] = '-gencode arch=compute_75,code=sm_75 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_89,code=sm_89'
    build_cmd = ['cargo', 'build', '--release', '--bin', 'gecko', '--features', 'cuda']
else:
    build_cmd = ['cargo', 'build', '--release', '--bin', 'gecko']

print('Running:', ' '.join(build_cmd))
subprocess.run(build_cmd, env=env, check=True)
!./target/release/gecko --version
!./target/release/gecko backends

# Also build C `gecko_fast` from `gpu-backends` for head-to-head C vs. Rust CPU comparison
!git clone --depth 1 -b gpu-backends {REPO_URL} /content/gecko_c
subprocess.run([
    'gcc', '-O3', '-D_FILE_OFFSET_BITS=64', '-march=native', '-pthread',
    '/content/gecko_c/src/gecko_fast.c',
    '/content/gecko_c/src/comparisonFunctions.c',
    '/content/gecko_c/src/commonFunctions.c',
    '-lm', '-o', '/content/gecko_c/gecko_fast',
], check=True)
print('Built C gecko_fast -> /content/gecko_c/gecko_fast')

In [ ]:
# 3. Configure whole-genome Human vs. Chimpanzee benchmark and auto-detect available backends
import os, subprocess

GECKO_BIN = '/content/gecko/target/release/gecko'
C_FAST_BIN = '/content/gecko_c/gecko_fast'
WORK_DIR = '/content/wg_bench'
RESULTS_JSON = '/content/wg_benchmark_results.json'
os.makedirs(WORK_DIR, exist_ok=True)

# Comparison parameters (matching our 24-chromosome Human GRCh38.p14 vs. Chimp Clint_PTRv2 report)
WORD_LENGTH = 32
MIN_LENGTH = 1000
MIN_SIMILARITY = 75
FREQ_LIMIT = 20
MEMORY_BUDGET = '4G'  # Fits all 24 chromosomes 100% in RAM with 2-bit PackedSeq (0 disk spill)

# Full 24 homologous chromosome pairs (chr2 pairs Human chr2 against Chimp chr2A + chr2B).
# Tip: for a 1-minute smoke test, slice e.g. CHROMOSOMES = ['chr21', 'chr22', 'chr1']
CHROMOSOMES = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']

# Include C `gecko_fast` (from `gpu-backends`) alongside Rust backends
INCLUDE_C_FAST = True
# Set RESUME = True to keep completed chromosomes if Colab reconnects mid-run
RESUME = False
# Set CHECK_PARITY = True to verify every backend's fragment recall against Rust CPU
CHECK_PARITY = True

# Auto-detect available Rust backends from `gecko backends`
probe_out = subprocess.run([GECKO_BIN, 'backends'], capture_output=True, text=True, check=True).stdout
print(probe_out)

BACKENDS = []
for b in ['cpu', 'wgpu', 'cuda']:
    for line in probe_out.splitlines():
        if line.strip().startswith(f'{b} ') and 'AVAILABLE' in line:
            BACKENDS.append(b)
            break
if INCLUDE_C_FAST and os.path.exists(C_FAST_BIN):
    BACKENDS.insert(1, 'c_fast')

# Optional manual override, e.g. BACKENDS = ['cpu', 'c_fast', 'cuda']
print('Selected backends:', BACKENDS)
print(f'Chromosomes ({len(CHROMOSOMES)}):', ', '.join(CHROMOSOMES))

In [ ]:
# 4. Run full 24-chromosome Human (GRCh38.p14) vs. Chimpanzee (Clint_PTRv2) benchmark across all backends
import gzip, json, os, re, shutil, struct, subprocess, tempfile, time

HS_BASE = (
    'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/'
    'GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_assembly_structure/'
    'Primary_Assembly/assembled_chromosomes/FASTA'
)
PT_BASE = (
    'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/002/880/755/'
    'GCF_002880755.1_Clint_PTRv2/GCF_002880755.1_Clint_PTRv2_assembly_structure/'
    'Primary_Assembly/assembled_chromosomes/FASTA'
)

def curl_download(url, dst):
    subprocess.run([
        'curl', '-fsSL',
        '--connect-timeout', '15',
        '--speed-time', '20',
        '--speed-limit', '50000',
        '--retry', '5',
        '--retry-delay', '2',
        '--retry-all-errors',
        '-o', dst, url,
    ], check=True)

def gunzip_to(src_gz, dst_fa, mode='wb'):
    with gzip.open(src_gz, 'rb') as fin, open(dst_fa, mode) as fout:
        shutil.copyfileobj(fin, fout, 1 << 20)

def download_pair(ch):
    hs_fa = os.path.join(WORK_DIR, f'hs_{ch}.fna')
    pt_fa = os.path.join(WORK_DIR, f'pt_{ch}.fna')
    hs_gz = hs_fa + '.gz'
    curl_download(f'{HS_BASE}/{ch}.fna.gz', hs_gz)
    gunzip_to(hs_gz, hs_fa, 'wb')
    os.remove(hs_gz)

    if ch == 'chr2':
        for part, mode in [('chr2A', 'wb'), ('chr2B', 'ab')]:
            p_gz = os.path.join(WORK_DIR, f'pt_{part}.fna.gz')
            curl_download(f'{PT_BASE}/{part}.fna.gz', p_gz)
            gunzip_to(p_gz, pt_fa, mode)
            os.remove(p_gz)
    else:
        pt_gz = pt_fa + '.gz'
        curl_download(f'{PT_BASE}/{ch}.fna.gz', pt_gz)
        gunzip_to(pt_gz, pt_fa, 'wb')
        os.remove(pt_gz)
    return hs_fa, pt_fa

def parse_frags(path):
    frags = []
    with open(path, 'rb') as f:
        n0, n1 = struct.unpack('>QQ', f.read(16))
        while len(b := f.read(93)) == 93:
            diag, xs, ys, xe, ye, length, ident, score, sim, sx, sy, blk, st = struct.unpack('>qQQQQQQQfQQqB', b)
            ch = chr(st)
            if ch == 'r':
                ys = n1 - ys - 1
                ye = n1 - ye - 1
            frags.append((xs, ys, xe, ye, ch, length, score, ident, sx, sy))
    return frags

def compute_parity(ref_frags, test_frags):
    sr, st = set(ref_frags), set(test_frags)
    if not sr:
        return 100.0, 100.0
    exact = len(sr & st)
    by_diag = {}
    for fr in test_frags:
        dk = (fr[4], fr[0] - fr[1] if fr[4] == 'f' else fr[0] + fr[1], fr[8], fr[9])
        by_diag.setdefault(dk, []).append((fr[0], fr[2]))
    cov = 0
    for fr in (sr - st):
        dk = (fr[4], fr[0] - fr[1] if fr[4] == 'f' else fr[0] + fr[1], fr[8], fr[9])
        for a, b in by_diag.get(dk, ()):
            if min(fr[2], b) - max(fr[0], a) + 1 >= 0.9 * (fr[2] - fr[0] + 1):
                cov += 1
                break
    return 100.0 * exact / len(sr), 100.0 * (exact + cov) / len(sr)

def run_backend(backend, ch, hs_fa, pt_fa):
    outdir = os.path.join(WORK_DIR, f'out_{ch}_{backend}')
    os.makedirs(outdir, exist_ok=True)
    if backend == 'c_fast':
        prefix_path = os.path.join(outdir, ch)
        threads = str(os.cpu_count() or 2)
        cmd = [
            C_FAST_BIN, hs_fa, pt_fa, prefix_path,
            str(MIN_LENGTH), str(MIN_SIMILARITY), str(WORD_LENGTH), '1',
            threads, str(FREQ_LIMIT),
        ]
    else:
        cmd = [
            GECKO_BIN, 'compare', hs_fa, pt_fa,
            '-o', outdir,
            '--prefix', ch,
            '--word-length', str(WORD_LENGTH),
            '-l', str(MIN_LENGTH),
            '-s', str(MIN_SIMILARITY),
            '--freq-limit', str(FREQ_LIMIT),
            '-m', MEMORY_BUDGET,
            '--backend', backend,
            '--format', 'frags',
            '--json-stats',
        ]
    with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
        t0 = time.perf_counter()
        p = subprocess.Popen(cmd, stdout=out_f, stderr=err_f)
        _, status, rusage = os.wait4(p.pid, 0)
        wall_s = time.perf_counter() - t0
        out_f.seek(0)
        err_f.seek(0)
        stdout = out_f.read().decode('utf-8', errors='replace')
        stderr = err_f.read().decode('utf-8', errors='replace')
        rss_mib = rusage.ru_maxrss / 1024.0

    if status != 0:
        shutil.rmtree(outdir, ignore_errors=True)
        err_lines = (stderr.strip() or stdout.strip()).splitlines()
        err_msg = err_lines[-1] if err_lines else f'exit {status}'
        return {'ok': False, 'error': err_msg}, None

    frags_path = os.path.join(outdir, f'{ch}.frags')
    frags = parse_frags(frags_path) if CHECK_PARITY else None

    if backend == 'c_fast':
        combined = stdout + '\n' + stderr
        m_seq = re.search(r'SeqX:\s*(\d+)\s*bp.*SeqY:\s*(\d+)\s*bp', combined)
        m_frag = re.search(r'Fragments:\s*(\d+)\s*\(F:(\d+)\s*R:(\d+)\)', combined)
        m_time = re.search(
            r'Timing:\s*load=([0-9.]+)s\s+index=([0-9.]+)s\s+query\+ext=([0-9.]+)s\s+write=([0-9.]+)s',
            combined,
        )
        load_s = float(m_time.group(1)) if m_time else 0.0
        compute_s = (float(m_time.group(2)) + float(m_time.group(3))) if m_time else wall_s
        write_s = float(m_time.group(4)) if m_time else 0.0
        n_frags = int(m_frag.group(1)) if m_frag else (len(frags) if frags is not None else 0)
        n_fwd = int(m_frag.group(2)) if m_frag else 0
        n_rev = int(m_frag.group(3)) if m_frag else 0
        q_bases = int(m_seq.group(1)) if m_seq else 0
        r_bases = int(m_seq.group(2)) if m_seq else 0
        shutil.rmtree(outdir, ignore_errors=True)
        return {
            'ok': True,
            'backend_label': f'c_fast (C AVX2, {os.cpu_count() or 2} threads)',
            'wall_s': round(wall_s, 3),
            'load_s': round(load_s, 3),
            'compute_s': round(compute_s, 3),
            'write_s': round(write_s, 3),
            'rss_mib': round(rss_mib, 1),
            'fragments': n_frags,
            'forward_fragments': n_fwd,
            'reverse_fragments': n_rev,
            'spilled_runs': 0,
            'query_bases': q_bases,
            'reference_bases': r_bases,
        }, frags

    st = json.loads(stdout)
    shutil.rmtree(outdir, ignore_errors=True)
    return {
        'ok': True,
        'backend_label': st.get('backend', backend),
        'wall_s': round(wall_s, 3),
        'load_s': round(st.get('load_secs', 0.0), 3),
        'compute_s': round(st.get('compute_secs', wall_s), 3),
        'write_s': round(st.get('write_secs', 0.0), 3),
        'rss_mib': round(rss_mib, 1),
        'fragments': st['fragments'],
        'forward_fragments': st['forward_fragments'],
        'reverse_fragments': st['reverse_fragments'],
        'spilled_runs': st['spilled_runs'],
        'query_bases': st['query_bases'],
        'reference_bases': st['reference_bases'],
    }, frags

# Load existing results when resuming
results = {}
if RESUME and os.path.exists(RESULTS_JSON):
    with open(RESULTS_JSON) as f:
        results = json.load(f)
    print(f'Resuming with {len(results)} chromosomes already recorded.')

for idx, ch in enumerate(CHROMOSOMES, 1):
    existing = results.get(ch, {})
    needed = [b for b in BACKENDS if b not in existing]
    if not needed:
        print(f'[{idx:2d}/{len(CHROMOSOMES)}] {ch}: already complete, skipping.')
        continue

    print(f'\n[{idx:2d}/{len(CHROMOSOMES)}] {ch} — downloading FASTAs...', flush=True)
    hs_fa, pt_fa = download_pair(ch)

    # If checking parity and CPU already cached in JSON, re-run CPU only if a GPU backend needs parity
    run_list = list(needed)
    if CHECK_PARITY and 'cpu' not in run_list and any(b != 'cpu' for b in run_list):
        run_list = ['cpu'] + run_list

    cpu_frags = None
    row = dict(existing)
    for b in run_list:
        rec, frags = run_backend(b, ch, hs_fa, pt_fa)
        if b == 'cpu' and rec['ok']:
            cpu_frags = frags
        elif CHECK_PARITY and rec['ok'] and cpu_frags is not None and frags is not None:
            ex_pct, rec_pct = compute_parity(cpu_frags, frags)
            rec['exact_pct'] = round(ex_pct, 3)
            rec['recall_pct'] = round(rec_pct, 4)

        if b in needed:
            row[b] = rec

        if rec['ok']:
            parity_str = f" | recall={rec['recall_pct']:.3f}%" if 'recall_pct' in rec else ''
            print(
                f"  [{b:<6}] wall={rec['wall_s']:6.2f}s "
                f"(load={rec['load_s']:.2f}s compute={rec['compute_s']:.2f}s) | "
                f"RSS={rec['rss_mib']:6.1f} MiB | frags={rec['fragments']:,} | "
                f"spilled={rec['spilled_runs']}{parity_str}",
                flush=True,
            )
        else:
            print(f"  [{b:<6}] SKIPPED/ERROR: {rec['error']}", flush=True)

    results[ch] = row
    with open(RESULTS_JSON, 'w') as f:
        json.dump(results, f, indent=2)

    # Immediately free FASTA disk space before moving to the next chromosome
    os.remove(hs_fa)
    os.remove(pt_fa)

print('\nAll chromosomes finished! Saved raw JSON to:', RESULTS_JSON)

In [ ]:
# 5. Compile full per-chromosome & genome-wide timing, speedup, RSS, and parity table
import json
import pandas as pd
from IPython.display import display, Markdown

with open(RESULTS_JSON) as f:
    results = json.load(f)

rows = []
for ch in CHROMOSOMES:
    r = results.get(ch, {})
    any_ok = next((r[b] for b in BACKENDS if b in r and r[b].get('ok')), None)
    entry = {
        'Chr': 'chr2 (2A+2B)' if ch == 'chr2' else ch,
        'Human_Mb': round(any_ok['query_bases'] / 1e6, 2) if any_ok else None,
        'Chimp_Mb': round(any_ok['reference_bases'] / 1e6, 2) if any_ok else None,
    }
    cpu_wall = r.get('cpu', {}).get('wall_s') if r.get('cpu', {}).get('ok') else None
    for b in BACKENDS:
        br = r.get(b, {})
        if br.get('ok'):
            entry[f'{b}_wall_s'] = br['wall_s']
            entry[f'{b}_compute_s'] = br['compute_s']
            entry[f'{b}_rss_MiB'] = br['rss_mib']
            entry[f'{b}_frags'] = br['fragments']
            if b != 'cpu' and cpu_wall:
                entry[f'{b}_speedup'] = round(cpu_wall / br['wall_s'], 2)
            if 'recall_pct' in br:
                entry[f'{b}_recall_%'] = br['recall_pct']
        else:
            entry[f'{b}_wall_s'] = None
            entry[f'{b}_compute_s'] = None
            entry[f'{b}_rss_MiB'] = None
            entry[f'{b}_frags'] = None
            if b != 'cpu':
                entry[f'{b}_speedup'] = None
                entry[f'{b}_recall_%'] = None
    rows.append(entry)

df = pd.DataFrame(rows)

# Build TOTAL summary row
total = {
    'Chr': 'TOTAL',
    'Human_Mb': round(df['Human_Mb'].sum(), 2),
    'Chimp_Mb': round(df['Chimp_Mb'].sum(), 2),
}
for b in BACKENDS:
    valid_mask = df[f'{b}_wall_s'].notna()
    n_ok = int(valid_mask.sum())
    b_tot = df[f'{b}_wall_s'].sum()
    total[f'{b}_wall_s'] = round(b_tot, 2) if n_ok > 0 else None
    total[f'{b}_compute_s'] = round(df[f'{b}_compute_s'].sum(), 2) if n_ok > 0 else None
    total[f'{b}_rss_MiB'] = round(df[f'{b}_rss_MiB'].max(), 1) if n_ok > 0 else None
    total[f'{b}_frags'] = int(df[f'{b}_frags'].sum()) if n_ok > 0 else None
    if b != 'cpu' and n_ok > 0 and 'cpu_wall_s' in df:
        # Compare speedup strictly over the chromosomes completed by BOTH backends
        cpu_matched = df.loc[valid_mask, 'cpu_wall_s'].sum()
        total[f'{b}_speedup'] = round(cpu_matched / b_tot, 2) if b_tot > 0 else None
        if f'{b}_recall_%' in df:
            total[f'{b}_recall_%'] = round(df.loc[valid_mask, f'{b}_recall_%'].mean(), 4)

df_full = pd.concat([df, pd.DataFrame([total])], ignore_index=True)

# Export CSV and Markdown
csv_path = '/content/human_vs_chimp_backend_timings.csv'
md_path = '/content/human_vs_chimp_backend_timings.md'
df_full.to_csv(csv_path, index=False)

def fmt_val(col, v):
    if pd.isna(v):
        return '—'
    if col.endswith('_s'):
        return f'{v:.2f}s'
    if col.endswith('_MiB'):
        return f'{v:,.1f}'
    if col.endswith('_frags'):
        return f'{int(v):,}'
    if col.endswith('_speedup'):
        return f'{v:.2f}x'
    if col.endswith('_recall_%'):
        return f'{v:.3f}%'
    if col.endswith('_Mb'):
        return f'{v:.2f}'
    return str(v)

cols = list(df_full.columns)
md_lines = [
    f'## Whole Human (`GRCh38.p14`) vs. Chimpanzee (`Clint_PTRv2`) Backend Timings (`WL={WORD_LENGTH}, L={MIN_LENGTH}, S={MIN_SIMILARITY}%, FL={FREQ_LIMIT}`)',
    '',
    '| ' + ' | '.join(cols) + ' |',
    '| ' + ' | '.join([':---' if c == 'Chr' else '---:' for c in cols]) + ' |',
]
for _, r in df_full.iterrows():
    is_tot = r['Chr'] == 'TOTAL'
    vals = [f'**{fmt_val(c, r[c])}**' if is_tot else fmt_val(c, r[c]) for c in cols]
    md_lines.append('| ' + ' | '.join(vals) + ' |')

md_text = '\n'.join(md_lines)
with open(md_path, 'w') as f:
    f.write(md_text + '\n')

display(Markdown(md_text))
print(f'\nSaved table to {csv_path} and {md_path}')

In [ ]:
# 6. Plot per-chromosome and total whole-genome runtime across backends
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={'width_ratios': [3, 1.2]})
colors = {'cpu': '#2563eb', 'c_fast': '#ef4444', 'wgpu': '#f59e0b', 'cuda': '#10b981'}

x = np.arange(len(df))
w = 0.8 / max(len(BACKENDS), 1)
for i, b in enumerate(BACKENDS):
    vals = df[f'{b}_wall_s'].fillna(0).values
    ax1.bar(x + (i - (len(BACKENDS) - 1) / 2) * w, vals, width=w, label=b.upper(), color=colors.get(b, None))

ax1.set_xticks(x)
ax1.set_xticklabels([c.replace(' (2A+2B)', '*') for c in df['Chr']], rotation=45, ha='right')
ax1.set_ylabel('Wall-clock time (s)')
ax1.set_title(f'Per-Chromosome Wall Time (Human GRCh38 vs Chimp, WL={WORD_LENGTH})')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

tot_labels = [b.upper() for b in BACKENDS]
tot_mins = [(total[f'{b}_wall_s'] or 0) / 60.0 for b in BACKENDS]
bars = ax2.bar(tot_labels, tot_mins, color=[colors.get(b, None) for b in BACKENDS])
ax2.set_ylabel('Total whole-genome time (min)')
ax2.set_title('24-Chromosome Total Runtime')
ax2.grid(axis='y', alpha=0.3)
for bar, b, m in zip(bars, BACKENDS, tot_mins):
    sp = f" ({total[f'{b}_speedup']:.2f}x)" if b != 'cpu' and total.get(f'{b}_speedup') else ''
    ax2.text(bar.get_x() + bar.get_width() / 2, m, f'{m:.2f}m{sp}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('/content/human_vs_chimp_backend_speedup.png', dpi=150)
plt.show()